# Entity ***Tokens***
+ Silver layer
+ Priority 1
---
### Imports

In [0]:
%run ../common/medallion_functions

In [0]:
from pyspark.sql.functions import current_timestamp

### Parameters

In [0]:
# layer = dbutils.widgets.get("layer")
# entity_name = dbutils.widgets.get("entity_name")
# load_pattern = dbutils.widgets.get("load_pattern")
layer = "silver"
entity_name = "tokens"
load_pattern = "1"

### Execution

In [0]:
bz_tokens_df = spark.read.table(f"uniswap.bronze.{entity_name}")

In [0]:
display(bz_tokens_df)

In [0]:
sv_pools_df = spark.sql(f"""
    SELECT
        id AS token_id,
        symbol,
        name,
        CAST(decimals AS INT) AS decimals,
        CAST(totalSupply / POW(10, CAST(decimals AS INT)) AS DOUBLE) AS total_supply,
        CAST(derivedETH AS DOUBLE) AS derived_price_ETH,
        ROUND(
            CAST(feesUSD AS DOUBLE),
            2
        ) AS fees_USD,
        ROUND(
            CAST(totalValueLockedUSD AS DOUBLE),
            2
        ) AS total_value_locked_USD,
        CAST(totalValueLocked AS DOUBLE) AS total_value_locked_token,
        ROUND(
            CAST(volumeUSD AS DOUBLE),
            2
        ) AS volume_USD,
        CAST(volume AS DOUBLE) AS volume_token,
        CAST(txCount AS INT) AS tx_count,
        CAST(poolCount AS INT) AS pool_count,
        CURRENT_TIMESTAMP() AS load_timestamp
    FROM {{df}}
""", df = bz_tokens_df)

### Save & exit

In [0]:
load_result = load_entity(sv_pools_df,
                        entity_name,
                        load_pattern,
                        layer
                        #,
                        )

In [0]:
dbutils.notebook.exit(load_result)